In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [ ]:
def collect_h5_file_paths(root_dir):
    file_paths = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith('h5'):
                file_paths.append(os.path.join(dirpath, filename))

    return file_paths

In [ ]:
ROOT_DIR = './data'
file_paths = collect_h5_file_paths(ROOT_DIR)

In [ ]:
len(file_paths)

In [ ]:
def extract_features_from_h5(h5_file_path):
    """
    從 Million Song Dataset 的 .h5 檔案中提取歌曲 ID 和 24 維音色特徵向量 (Mean/Std)。
    
    參數:
        h5_file_path (str): HDF5 檔案的完整路徑。

    返回:
        tuple (str, numpy.ndarray) 或 (None, None): 
        包含 (song_id, feature_vector) 或 (None, None)。
    """
    try:
        with h5py.File(h5_file_path, 'r') as f:
            # 檢查關鍵群組是否存在
            if 'metadata' not in f or 'analysis' not in f:
                raise KeyError("Required groups 'metadata' or 'analysis' not found.")
            
            # 1. 提取歌曲 ID (從複合數據集 'songs' 中)
            # 假設 'songs' 是唯一的複合數據集，且 'song_id' 在其中
            songs_metadata = f['metadata']['songs']
            # 提取第一個記錄中的 'song_id' 欄位，並將其解碼為 Python 字串
            song_id = songs_metadata['song_id'][0].decode('utf-8')

            # 2. 提取 segments_timbre 矩陣
            # 這是位於 'analysis' 群組下的數據集
            timbre = f['analysis']['segments_timbre'][:]
            
            # 3. 彙總計算 (Mean and Std)
            # Timbre 應有 12 維。計算每維的平均值 (12 維) 和標準差 (12 維)。
            timbre_mean = np.mean(timbre, axis=0)
            timbre_std = np.std(timbre, axis=0)
            
            if len(timbre_mean) != 12:
                 raise ValueError(f"Timbre dimension mismatch: expected 12, got {len(timbre_mean)}")

            # 4. 拼接成單一 24 維特徵向量
            feature_vector = np.concatenate([timbre_mean, timbre_std])
            
            return song_id, feature_vector

    except KeyError as ke:
        # 捕獲因路徑錯誤或數據集/欄位缺失導致的錯誤
        print(f"Skipping error processing {h5_file_path}: Key error: {ke}")
        return None, None
    except IndexError as ie:
        # 捕獲因嘗試對空數據集進行索引 [0] 導致的錯誤
        print(f"Skipping error processing {h5_file_path}: Index error: {ie}")
        return None, None
    except Exception as e:
        # 捕獲其他所有異常
        print(f"Skipping error processing {h5_file_path}: General error: {e}")
        return None, None

In [ ]:
all_song_data = {}

# 遍歷所有檔案並提取數據
print("Starting to scan files and extract features...")
for path in file_paths:
    song_id, features = extract_features_from_h5(path)
    
    if song_id:
        all_song_data[song_id] = features
print(f"Finished scanning. Total {len(all_song_data)} songs processed.")

In [ ]:
all_song_data